In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from pgmpy.estimators import BIC, BayesianEstimator, HillClimbSearch
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.dpi"] = 150
plt.style.use("ggplot")


# Load data in a DataFrame
data = pd.read_excel("./Dataset1_BankClients.xlsx")
# Drop the column by its actual name (e.g., 'ID' or the actual name of the column)
data = data.drop(columns=["ID"])  # Replace 'ID' with the actual column name to drop

# Define Business Mapping dictionary
mapping_dicts = {
    "Gender": {"0": "Male", "1": "Female", "0.0": "Male", "1.0": "Female"},
    "Job": {
        "1": "Unemployed",
        "2": "Employee/Worker",
        "3": "Manager/Executive",
        "4": "Entrepreneur/Freelancer",
        "5": "Retired",
    },
    "Area": {"1": "Nord", "2": "Centro", "3": "Sud/Isole"},
    "CitySize": {"1": "Small town", "2": "Medium-sized city", "3": "Large city"},
    "Investments": {
        "1": "No investments",
        "2": "Mostly lump sum",
        "3": "Capital accumulation",
    },
}

## Data processing


### One hot encoding


In [ ]:
# Specify categorical variables
categorical_columns = ["Gender", "Job", "Area", "CitySize", "Investments"]

# Apply one-hot encoding to categorical features
data_encoded = pd.get_dummies(data, columns=categorical_columns, dtype=int)

### Scaling


In [ ]:
# Identify continuous numeric columns from the original data (exclude categorical columns).
continuous_columns = [
    col
    for col in data.select_dtypes(include=["number"]).columns
    if col not in categorical_columns
]

# Scale only continuous columns and keep one-hot columns unchanged.
scaler = StandardScaler()
data_scaled = data_encoded.copy()
data_scaled[continuous_columns] = scaler.fit_transform(data_scaled[continuous_columns])

data_scaled.head()

## Clustering


### Adaptive Clustering with Bayesian Gaussian Mixture (BGM)

- Set an upper bound of `K = 15` components.
- Let the Dirichlet Process prior auto-prune unsupported clusters.
- Keep only clusters with meaningful weights (default threshold: `> 5%`) as final Persona frameworks.


In [ ]:
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture

# Set upper limit of candidate clusters
max_clusters = 15

# Train BGM with Dirichlet Process prior for auto-pruning
bgm = BayesianGaussianMixture(
    n_components=max_clusters,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    random_state=42,
    max_iter=1000,
    init_params="kmeans",
)

bgm.fit(data_scaled)

In [ ]:
# Raw cluster assignment from all components
raw_labels = bgm.predict(data_scaled)
raw_weights = pd.Series(bgm.weights_, name="weight")

# Lock final personas by removing low-weight components
weight_threshold = 0.05  # only keep components with > 5% weight
active_components = raw_weights[raw_weights > weight_threshold].index.tolist()

# Keep only samples assigned to active components
active_mask = pd.Series(raw_labels).isin(active_components)
data_persona = data_scaled.loc[active_mask].copy()
data_persona["persona_cluster"] = pd.Series(raw_labels)[active_mask].values

# Re-map cluster ids to consecutive labels (0..n-1) for readability
cluster_remap = {old: new for new, old in enumerate(sorted(active_components))}
data_persona["persona_cluster"] = data_persona["persona_cluster"].map(cluster_remap)

# Summary table of retained persona frameworks
persona_summary = (
    data_persona["persona_cluster"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
    .to_frame()
)
persona_summary["count"] = (
    data_persona["persona_cluster"].value_counts().sort_index().values
)

print(f"Max components (K): {max_clusters}")
print(f"Retained persona frameworks: {len(active_components)}")
print(f"Active original component IDs: {sorted(active_components)}")

persona_summary

## Basic Profiling & Naming


### Qualitative Overlay: Rebuild Centroid Features into Business Tags

- Recover continuous centroid means in original units.
- Decode dominant one-hot categories for each retained cluster.
- Produce business-readable profiles to support persona naming.


In [ ]:
# Cluster means on scaled data
cluster_scaled_means = data_persona.groupby("persona_cluster").mean(numeric_only=True)
# Restore continuous means to original units
continuous_scaled = cluster_scaled_means[continuous_columns]
continuous_original = pd.DataFrame(
    scaler.inverse_transform(continuous_scaled),
    columns=continuous_columns,
    index=continuous_scaled.index,
)
# Decode dominant category from one-hot proportions
categorical_profile = pd.DataFrame(index=cluster_scaled_means.index)

for col in categorical_columns:
    onehot_cols = [c for c in cluster_scaled_means.columns if c.startswith(f"{col}_")]
    if not onehot_cols:
        continue

    proportions = cluster_scaled_means[onehot_cols]
    dominant_onehot = proportions.idxmax(axis=1)
    dominant_value_raw = dominant_onehot.str.replace(f"{col}_", "", regex=False)

    dominant_value_mapped = dominant_value_raw.map(
        lambda x: mapping_dicts.get(col, {}).get(str(x), x)
    )

    dominant_share = proportions.max(axis=1)

    categorical_profile[f"{col}_dominant"] = dominant_value_mapped
    categorical_profile[f"{col}_share"] = dominant_share

# Build qualitative business tags
business_profile = continuous_original.join(categorical_profile)

business_profile

In [ ]:
def age_band(age_value: float) -> str:
    if age_value < 35:
        return "18-34"
    if age_value < 55:
        return "35-54"
    if age_value <= 70:
        return "55-70"
    return "70+"


def pct_to_tag(val: float, high_thresh=0.66, low_thresh=0.33) -> str:
    if val >= high_thresh:
        return "High"
    if val <= low_thresh:
        return "Low"
    return "Medium"

In [ ]:
business_profile["Age_band"] = business_profile["Age"].apply(age_band)
business_profile["Wealth_tag"] = business_profile["Wealth"].apply(
    lambda x: pct_to_tag(x, 0.7, 0.3) + " Wealth"
)
business_profile["Debt_tag"] = business_profile["Debt"].apply(
    lambda x: pct_to_tag(x, 0.7, 0.3) + " Debt"
)
business_profile["Digital_tag"] = business_profile["Digital"].apply(
    lambda x: pct_to_tag(x, 0.6, 0.4) + " Digital"
)

extra_percentile_features = [
    "Income",
    "FinEdu",
    "ESG",
    "BankFriend",
    "LifeStyle",
    "Luxury",
    "Saving",
]

for feat in extra_percentile_features:
    if feat in business_profile.columns:
        business_profile[f"{feat}_tag"] = business_profile[feat].apply(
            lambda x: pct_to_tag(x, 0.66, 0.33) + f" {feat}"
        )

# Building the Qualitative Overlay
business_profile["Qualitative_overlay"] = (
    "Age "
    + business_profile["Age_band"].astype(str)
    + " | "
    + business_profile.get("Gender_dominant", "").astype(str)
    + " | "
    + business_profile.get("Job_dominant", "").astype(str)
    + " | "
    + business_profile.get("Wealth_tag", "").astype(str)
    + " | Edu: "
    + business_profile.get("FinEdu_tag", "").astype(str).str.replace(" FinEdu", "")
    + " | Lux: "
    + business_profile.get("Luxury_tag", "").astype(str).str.replace(" Luxury", "")
)

In [ ]:
# Optional: manual business naming column to fill in after review
business_profile["Suggested_name"] = ""

# Display high-value columns first
display_cols = [
    "Age_band",
    "Gender_dominant",
    "Job_dominant",
    "Investments_dominant",
    # Financial and Liability Status
    "Wealth_tag",
    "Income_tag",
    "Debt_tag",
    "Saving_tag",
    # Cognitive and Digital Preferences
    "Digital_tag",
    "FinEdu_tag",
    # Consumption and Values
    "Luxury_tag",
    "LifeStyle_tag",
    "ESG_tag",
    # Social and Relational Context
    "BankFriend_tag",
    # Demographic and Geographic Context
    "CitySize_dominant",
    "Area_dominant",
    "Qualitative_overlay",
    "Suggested_name",
]

existing_cols = [c for c in display_cols if c in business_profile.columns]
business_profile[existing_cols]


In [ ]:
business_profile["Qualitative_overlay"]

## Introducing Bayesian Networks for Deep Insight and Inference


In [ ]:
# 1) Select a target persona subgroup (closest to "Wealthy Widow" profile)
profile = business_profile.copy()
profile["_age_score"] = profile["Age"]
profile["_debt_score"] = -profile["Debt"]
profile["_wealth_score"] = profile["Wealth"]

if "Gender_dominant" in profile.columns:
    profile["_gender_score"] = (profile["Gender_dominant"].astype(str) == "1").astype(
        int
    )
else:
    profile["_gender_score"] = 0

profile["_persona_score"] = (
    0.4 * profile["_age_score"].rank(pct=True)
    + 0.3 * profile["_debt_score"].rank(pct=True)
    + 0.2 * profile["_wealth_score"].rank(pct=True)
    + 0.1 * profile["_gender_score"].rank(pct=True)
)

wealthy_widow_cluster = int(profile["_persona_score"].idxmax())
persona_idx = data_persona[
    data_persona["persona_cluster"] == wealthy_widow_cluster
].index
persona_raw = data.loc[persona_idx].copy()

# 2) Build a discrete dataset for BN structure learning
bn_data = pd.DataFrame(index=persona_raw.index)

if "Age" in persona_raw.columns:
    bn_data["Age_band"] = pd.cut(
        persona_raw["Age"],
        bins=[-np.inf, 34, 54, 70, np.inf],
        labels=["18-34", "35-54", "55-70", "70+"],
    ).astype(str)

candidate_cats = [
    "Gender",
    "Job",
    "Area",
    "CitySize",
    "Investments",
    "MaritalStatus",
    "Mortgage",
]
for c in candidate_cats:
    if c in persona_raw.columns:
        bn_data[c] = persona_raw[c].astype(str)

if "Digital" in persona_raw.columns:
    digital_q = persona_raw["Digital"].quantile([0.33, 0.66]).values
    bn_data["Digital_level"] = pd.cut(
        persona_raw["Digital"],
        bins=[-np.inf, digital_q[0], digital_q[1], np.inf],
        labels=["Low", "Medium", "High"],
        include_lowest=True,
    ).astype(str)

if "Debt" in persona_raw.columns:
    debt_q = persona_raw["Debt"].quantile([0.33, 0.66]).values
    bn_data["Debt_level"] = pd.cut(
        persona_raw["Debt"],
        bins=[-np.inf, debt_q[0], debt_q[1], np.inf],
        labels=["Low", "Medium", "High"],
        include_lowest=True,
    ).astype(str)

# Need target for inference
need_derived_from_investments = False
if {"LowRiskInvestments", "HighRiskStocks"}.issubset(set(persona_raw.columns)):
    bn_data["Need"] = np.where(
        persona_raw["LowRiskInvestments"] >= persona_raw["HighRiskStocks"],
        "LowRiskInvestments",
        "HighRiskStocks",
    )
elif "Investments" in persona_raw.columns:
    inv_vals = persona_raw["Investments"]
    bn_data["Need"] = np.select(
        [inv_vals == inv_vals.min(), inv_vals == inv_vals.max()],
        ["LowRiskInvestments", "HighRiskStocks"],
        default="Balanced",
    )
    need_derived_from_investments = True
else:
    raise ValueError(
        "Need proxy missing. Add Investments or explicit risk-preference columns."
    )

# Avoid target leakage
if need_derived_from_investments and "Investments" in bn_data.columns:
    bn_data = bn_data.drop(columns=["Investments"])

bn_data = bn_data.dropna().copy()
if bn_data.empty:
    raise ValueError("No records available for BN after preprocessing.")

# Merge very rare states
for col in bn_data.columns:
    vc = bn_data[col].value_counts(normalize=True)
    rare = vc[vc < 0.02].index
    if len(rare) > 0:
        bn_data[col] = bn_data[col].replace(rare, "Other")

# pgmpy expects explicit categorical semantics for discrete structure learning.
for col in bn_data.columns:
    bn_data[col] = pd.Categorical(bn_data[col].astype(str))

if "Need" not in bn_data.columns:
    raise ValueError("Need variable is required for inference.")

# 3) Structure learning
hc = HillClimbSearch(bn_data)
learned_structure = hc.estimate(scoring_method=BIC(bn_data), max_indegree=3)
learned_edges = list(learned_structure.edges())

# If Need is isolated, add informative parents using mutual information.
need_incident = [e for e in learned_edges if "Need" in e]
if len(need_incident) == 0:
    predictors = [c for c in bn_data.columns if c != "Need"]
    if len(predictors) > 0:
        X = pd.DataFrame({c: pd.factorize(bn_data[c])[0] for c in predictors})
        y = pd.factorize(bn_data["Need"])[0]
        mi = mutual_info_classif(X, y, discrete_features=True, random_state=42)
        mi_series = pd.Series(mi, index=predictors).sort_values(ascending=False)
        top_predictors = mi_series[mi_series > 1e-8].head(3).index.tolist()
        if len(top_predictors) == 0:
            top_predictors = predictors[: min(2, len(predictors))]
        learned_edges.extend([(p, "Need") for p in top_predictors])

# Keep only nodes participating in the model to avoid CPD-missing errors.
model_nodes = sorted(
    set(["Need"] + [u for u, _ in learned_edges] + [v for _, v in learned_edges])
)
bn_train = bn_data[model_nodes].copy()

bn_model = DiscreteBayesianNetwork(learned_edges)
if "Need" not in bn_model.nodes():
    bn_model.add_node("Need")

bn_model.fit(
    bn_train, estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=10
)

# 4) Visualize learned DAG
plt.figure(figsize=(11, 7))
G = nx.DiGraph()
G.add_nodes_from(bn_model.nodes())
G.add_edges_from(bn_model.edges())
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=2500,
    node_color="#f5d76e",
    font_size=9,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=14,
)
plt.title(f"Learned Bayesian Network DAG | Persona Cluster {wealthy_widow_cluster}")
plt.axis("off")
plt.show()

# 5) Probabilistic inference
infer = VariableElimination(bn_model)

candidate_evidence = {}
if "Age_band" in bn_train.columns:
    candidate_evidence["Age_band"] = "55-70"
if "Mortgage" in bn_train.columns:
    mortgage_states = set(bn_train["Mortgage"].astype(str).unique())
    candidate_evidence["Mortgage"] = (
        "N" if "N" in mortgage_states else list(mortgage_states)[0]
    )
elif "Debt_level" in bn_train.columns:
    candidate_evidence["Debt_level"] = "Low"
if "Digital_level" in bn_train.columns:
    candidate_evidence["Digital_level"] = "Low"

# Keep only evidence variables present in the model with valid states.
evidence = {}
for k, v in candidate_evidence.items():
    if k in bn_model.nodes():
        cpd = bn_model.get_cpds(k)
        if cpd is not None and v in cpd.state_names[k]:
            evidence[k] = v

need_query = infer.query(variables=["Need"], evidence=evidence, show_progress=False)
need_prob = pd.Series(
    need_query.values, index=need_query.state_names["Need"], name="P(Need)"
).sort_values(ascending=False)

print(f"Selected persona cluster for BN analysis: {wealthy_widow_cluster}")
print(f"Subgroup size used for BN: {len(bn_train)}")
print("Need derived from Investments:", need_derived_from_investments)
print("Candidate evidence:", candidate_evidence)
print("Applied evidence:", evidence)
print("Learned edges:", list(bn_model.edges()))

need_prob.to_frame()